# VAE vs VQ-VAE Watermarking Performance Comparison (Improved)

## 실험 목표
1. **semantic_wm 데이터셋**으로 VAE와 VQ-VAE 학습
2. **Latent 차원**: 100D 고정
3. **개선사항**:
   - ✅ Decoder L2 정규화 적용
   - ✅ BatchNorm1d 사용
   - ✅ Beta = 0.01 (작은 값으로 클러스터 유지)
   - ✅ Dropout = 0.2
4. **평가 지표**: Watermark → Reconstructed CLIP Cosine Distance

## 비교 모델
```
1. VAE-100D (Improved):  L2 norm + BatchNorm + β=0.01
2. VQ-VAE-100D (Improved): L2 norm + BatchNorm + 512 codebook
```

## 파이프라인
```
Training:
  semantic_wm images → CLIP(512D) → VAE/VQ-VAE → Latent(100D)
  
Testing:
  Test image → CLIP(512D) → Latent(100D) → Watermark bits
            ↓
  Decoder → Reconstructed CLIP(512D) [L2 normalized]
            ↓
  Evaluation: Cosine Distance = 1 - Cosine Similarity
```

## 1. 환경 설정 및 패키지 설치

In [ ]:
# Core ML libraries
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Computer Vision & Image Processing
!pip install -q numpy>=2.0 scipy>=1.13 pillow==10.4.0 opencv-python
!pip install -q scikit-image scikit-learn matplotlib seaborn

In [ ]:
# Deep Learning utilities
!pip install -q einops==0.8.0 timm==0.9.12
!pip install -q tqdm easydict

In [ ]:
# Transformers and CLIP
!pip install -q transformers==4.45.2 open-clip-torch==2.26.1

In [ ]:
print("✅ All packages installed successfully!")

In [ ]:
# 런타임 재시작
import os
os.kill(os.getpid(), 9)

## 2. 라이브러리 Import

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
# semantic_wm 데이터셋 압축 해제
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("압축 해제 중...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("✅ 데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
print("\n=== 데이터셋 구조 ===")
!ls -R /content/semantic_wm/dataset/train/

## 4. 데이터셋 경로 설정

In [ ]:
# 데이터셋 경로
train_path = '/content/semantic_wm/dataset/train'
test_path = '/content/semantic_wm/dataset/test'

# 카테고리 및 최대 이미지 수
categories = ['normal', 'violence', 'sexual']
max_images_per_category = None  # None = 전체 사용

print("✅ 데이터셋 경로 설정 완료")
print(f"   - Train: {train_path}")
print(f"   - Test: {test_path}")
print(f"   - Categories: {categories}")

In [ ]:
# 데이터셋 경로 확인 (디버깅)
import os

print("\n" + "="*80)
print("데이터셋 경로 확인")
print("="*80)

# Train 경로 확인
if os.path.exists(train_path):
    print(f"✅ Train 경로 존재: {train_path}")
    train_subdirs = os.listdir(train_path)
    print(f"   하위 폴더/파일: {train_subdirs}")

    for category in categories:
        cat_path = os.path.join(train_path, category)
        if os.path.exists(cat_path):
            files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
            print(f"   [{category}] 이미지 파일 수: {len(files)}")
            if len(files) > 0:
                print(f"      예시: {files[:3]}")
        else:
            print(f"   ⚠️ [{category}] 경로 없음: {cat_path}")
else:
    print(f"❌ Train 경로가 존재하지 않습니다: {train_path}")
    print("\n가능한 대안 경로:")
    possible_paths = [
        './semantic_wm/dataset/train',
        '../semantic_wm/dataset/train',
        './dataset/train'
    ]
    for p in possible_paths:
        if os.path.exists(p):
            print(f"   ✅ {p}")

print("\n" + "-"*80)

# Test 경로 확인
if os.path.exists(test_path):
    print(f"✅ Test 경로 존재: {test_path}")
    test_subdirs = os.listdir(test_path)
    print(f"   하위 폴더/파일: {test_subdirs}")

    for category in categories:
        cat_path = os.path.join(test_path, category)
        if os.path.exists(cat_path):
            files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
            print(f"   [{category}] 이미지 파일 수: {len(files)}")
            if len(files) > 0:
                print(f"      예시: {files[:3]}")
        else:
            print(f"   ⚠️ [{category}] 경로 없음: {cat_path}")
else:
    print(f"❌ Test 경로가 존재하지 않습니다: {test_path}")

print("="*80)

## 5. CLIP 모델 로드

In [ ]:
# CLIP 모델 로드
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Evaluation mode
clip_model.eval()

print("✅ CLIP 모델 로드 완료")
print(f"   - Model: openai/clip-vit-base-patch32")
print(f"   - Device: {device}")

## 6. 데이터 로드 함수

In [ ]:
def load_images_from_category(base_path, category, max_images=None):
    """카테고리별 이미지 경로 로드"""
    category_path = os.path.join(base_path, category)

    if not os.path.exists(category_path):
        print(f"⚠️ 카테고리 경로가 존재하지 않습니다: {category_path}")
        return []

    image_paths = []

    for img_name in os.listdir(category_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            full_path = os.path.join(category_path, img_name)
            # 파일이 실제로 존재하고 크기가 0이 아닌지 확인
            if os.path.isfile(full_path) and os.path.getsize(full_path) > 0:
                image_paths.append(full_path)

    if max_images:
        image_paths = image_paths[:max_images]

    print(f"   [{category}] {len(image_paths)}개 이미지 발견")
    return image_paths


def extract_clip_embeddings(image_paths, model, processor, device, batch_size=32):
    """이미지 경로들로부터 CLIP embedding 추출"""
    embeddings = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting embeddings"):
        batch_paths = image_paths[i:i+batch_size]

        # 이미지 로드 시 에러 처리
        images = []
        valid_paths = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                images.append(img)
                valid_paths.append(p)
            except Exception as e:
                print(f"\n⚠️ 이미지 로드 실패: {p}")
                print(f"   Error: {str(e)}")
                continue

        if len(images) == 0:
            continue

        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
            # L2 정규화 (CLIP은 기본적으로 정규화된 embedding 생성)
            image_features = F.normalize(image_features, p=2, dim=1)

        embeddings.append(image_features.cpu().numpy())

    if len(embeddings) == 0:
        raise ValueError("유효한 이미지가 하나도 없습니다. 데이터셋 경로를 확인하세요.")

    return np.vstack(embeddings)


print("✅ 데이터 로드 함수 정의 완료")
print("   - load_images_from_category: 카테고리별 이미지 경로 로드")
print("   - extract_clip_embeddings: CLIP embedding 추출 (L2 정규화)")

## 7. Training 데이터 로드

In [ ]:
train_embeddings = []
train_labels = []
train_category_stats = {}

print("\n" + "="*80)
print("Training Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(train_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    train_embeddings.append(embeddings)
    train_labels.extend([category] * len(embeddings))
    train_category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
train_embeddings = np.vstack(train_embeddings)
train_labels = np.array(train_labels)

print("\n" + "="*80)
print("Training Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {train_embeddings.shape}")
print(f"전체 레이블 수: {len(train_labels)}")
print(f"평균 L2 norm: {np.linalg.norm(train_embeddings, axis=1).mean():.6f}")
print("\n카테고리별 통계:")
for cat, count in train_category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(train_labels)*100:.1f}%)")
print("="*80)

## 8. Test 데이터 로드

In [ ]:
test_embeddings = []
test_labels = []
test_category_stats = {}

print("\n" + "="*80)
print("Test Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(test_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    test_embeddings.append(embeddings)
    test_labels.extend([category] * len(embeddings))
    test_category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
test_embeddings = np.vstack(test_embeddings)
test_labels = np.array(test_labels)

print("\n" + "="*80)
print("Test Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {test_embeddings.shape}")
print(f"전체 레이블 수: {len(test_labels)}")
print(f"평균 L2 norm: {np.linalg.norm(test_embeddings, axis=1).mean():.6f}")
print("\n카테고리별 통계:")
for cat, count in test_category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(test_labels)*100:.1f}%)")
print("="*80)

## 9. VAE 모델 정의 (개선 버전)

In [ ]:
class ImprovedVAE(nn.Module):
    """
    Improved VAE for CLIP Embeddings

    개선사항:
    - BatchNorm1d 사용 (LayerNorm 대신)
    - Decoder 출력에 L2 정규화 적용
    - Dropout 0.2 (더 강한 정규화)
    """
    def __init__(self, input_dim=512, latent_dim=100):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder: CLIP(512) → Latent(latent_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: Latent(latent_dim) → CLIP'(512)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        recon = self.decoder(z)
        # ✅ L2 정규화 적용 (중요!)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

    def get_latent(self, x):
        """Inference용: latent 추출"""
        mu, _ = self.encode(x)
        return mu


print("✅ Improved VAE 모델 정의 완료")
print("   - Input: 512D (CLIP)")
print("   - Latent: 100D")
print("   - Output: 512D (L2 normalized)")
print("   - BatchNorm1d + Dropout 0.2")

## 10. VQ-VAE 모델 정의 (개선 버전)

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization Layer"""
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        # Codebook
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)

    def forward(self, z):
        # Calculate distances to all codebook vectors
        distances = torch.cdist(z, self.embeddings.weight)

        # Find nearest codebook vector
        encoding_indices = torch.argmin(distances, dim=1)

        # Get quantized vectors
        quantized = self.embeddings(encoding_indices)

        # VQ Loss
        codebook_loss = F.mse_loss(quantized, z.detach())
        commitment_loss = F.mse_loss(z, quantized.detach())
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss

        # Straight-through estimator
        quantized = z + (quantized - z).detach()

        return quantized, vq_loss, encoding_indices


class ImprovedVQVAE(nn.Module):
    """
    Improved VQ-VAE for CLIP Embeddings

    개선사항:
    - BatchNorm1d 사용
    - Decoder 출력에 L2 정규화 적용
    - Dropout 0.2
    """
    def __init__(self, input_dim=512, latent_dim=100, num_embeddings=512):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.num_embeddings = num_embeddings

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        # Vector Quantizer
        self.vq = VectorQuantizer(num_embeddings, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        recon = self.decoder(z)
        # ✅ L2 정규화 적용 (중요!)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        z = self.encode(x)
        quantized, vq_loss, encoding_indices = self.vq(z)
        recon = self.decode(quantized)
        return recon, vq_loss, encoding_indices

    def get_latent(self, x):
        """Inference용: quantized latent 추출"""
        z = self.encode(x)
        quantized, _, encoding_indices = self.vq(z)
        return quantized, encoding_indices


print("✅ Improved VQ-VAE 모델 정의 완료")
print("   - Input: 512D (CLIP)")
print("   - Latent: 100D (Discrete)")
print("   - Codebook: 512 entries")
print("   - Output: 512D (L2 normalized)")
print("   - BatchNorm1d + Dropout 0.2")

## 11. Loss 함수 정의

In [ ]:
def vae_loss(recon, target, mu, logvar, beta=0.001):
    """
    VAE Loss = Reconstruction Loss + β × KL Divergence

    Args:
        recon: Reconstructed CLIP embedding (L2 normalized)
        target: Original CLIP embedding (L2 normalized)
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        beta: Weight for KL divergence (작은 값으로 클러스터 유지)

    Returns:
        total_loss, recon_loss, kl_loss
    """
    # 1. Reconstruction loss (Cosine Distance) - CLIP에 최적화!
    cosine_sim = F.cosine_similarity(recon, target, dim=1).mean()
    recon_loss = 1 - cosine_sim  # Cosine Distance

    # 2. KL divergence
    # kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss


def vqvae_loss(recon, target, vq_loss):
    """
    VQ-VAE Loss = Reconstruction Loss + VQ Loss

    Args:
        recon: Reconstructed CLIP embedding (L2 normalized)
        target: Original CLIP embedding (L2 normalized)
        vq_loss: Vector quantization loss

    Returns:
        total_loss, recon_loss
    """
    # Reconstruction loss (Cosine Distance) - CLIP에 최적화!
    cosine_sim = F.cosine_similarity(recon, target, dim=1).mean()
    recon_loss = 1 - cosine_sim  # Cosine Distance

    # Total loss
    total_loss = recon_loss + vq_loss

    return total_loss, recon_loss


print("✅ Loss 함수 정의 완료")
print("   - vae_loss: Cosine Distance + β×KLD (β=0.01)")
print("   - vqvae_loss: Cosine Distance + VQ")

## 12. 데이터셋 및 DataLoader 준비

In [ ]:
# 데이터셋 클래스
class CLIPEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


# Train/Val split
X_train, X_val, y_train, y_val = train_test_split(
    train_embeddings, train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

# Dataset 생성
train_dataset = CLIPEmbeddingDataset(X_train, y_train)
val_dataset = CLIPEmbeddingDataset(X_val, y_val)

# DataLoader 생성
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("✅ 데이터셋 준비 완료")
print(f"   - 학습 데이터: {len(train_dataset)} samples")
print(f"   - 검증 데이터: {len(val_dataset)} samples")
print(f"   - Batch size: {batch_size}")
print(f"   - Train batches: {len(train_loader)}")
print(f"   - Val batches: {len(val_loader)}")

## 13. Training 함수

In [ ]:
def train_vae(model, train_loader, val_loader, epochs=50, lr=1e-3, beta=0.01):
    """VAE Training 함수"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_losses': [], 'train_recon': [], 'train_kld': [],
        'val_losses': [], 'val_recon': [], 'val_kld': [], 'val_cosine': []
    }

    best_val_loss = float('inf')

    # Epoch progress bar
    epoch_pbar = tqdm(range(epochs), desc="VAE Training")

    for epoch in epoch_pbar:
        # Training
        model.train()
        train_loss, train_recon, train_kld = 0, 0, 0

        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)

            # Forward
            recon, mu, logvar, _ = model(batch_x)
            loss, recon_loss, kl_loss = vae_loss(recon, batch_x, mu, logvar, beta)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_kld += kl_loss.item()

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_kld /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_recon, val_kld = 0, 0, 0
        val_cosine_sims = []

        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)

                recon, mu, logvar, _ = model(batch_x)
                loss, recon_loss, kl_loss = vae_loss(recon, batch_x, mu, logvar, beta)

                val_loss += loss.item()
                val_recon += recon_loss.item()
                val_kld += kl_loss.item()

                # Cosine similarity
                cos_sim = F.cosine_similarity(batch_x, recon, dim=1).mean()
                val_cosine_sims.append(cos_sim.item())

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_kld /= len(val_loader)
        val_cosine = np.mean(val_cosine_sims)

        # Save history
        history['train_losses'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_kld'].append(train_kld)
        history['val_losses'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_kld'].append(val_kld)
        history['val_cosine'].append(val_cosine)

        # Update epoch progress bar
        epoch_pbar.set_postfix({
            'train_loss': f"{train_loss:.4f}",
            'val_loss': f"{val_loss:.4f}",
            'cosine': f"{val_cosine:.4f}"
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), '/content/drive/MyDrive/semantic_wm/models/best_vae_model.pth')

    return history


def train_vqvae(model, train_loader, val_loader, epochs=50, lr=1e-3):
    """VQ-VAE Training 함수"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_losses': [], 'train_recon': [], 'train_vq': [],
        'val_losses': [], 'val_recon': [], 'val_vq': [], 'val_cosine': []
    }

    best_val_loss = float('inf')

    # Epoch progress bar
    epoch_pbar = tqdm(range(epochs), desc="VQ-VAE Training")

    for epoch in epoch_pbar:
        # Training
        model.train()
        train_loss, train_recon, train_vq = 0, 0, 0

        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)

            # Forward
            recon, vq_loss_val, _ = model(batch_x)
            loss, recon_loss = vqvae_loss(recon, batch_x, vq_loss_val)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_vq += vq_loss_val.item()

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_vq /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_recon, val_vq = 0, 0, 0
        val_cosine_sims = []

        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)

                recon, vq_loss_val, _ = model(batch_x)
                loss, recon_loss = vqvae_loss(recon, batch_x, vq_loss_val)

                val_loss += loss.item()
                val_recon += recon_loss.item()
                val_vq += vq_loss_val.item()

                # Cosine similarity
                cos_sim = F.cosine_similarity(batch_x, recon, dim=1).mean()
                val_cosine_sims.append(cos_sim.item())

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_vq /= len(val_loader)
        val_cosine = np.mean(val_cosine_sims)

        # Save history
        history['train_losses'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_vq'].append(train_vq)
        history['val_losses'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_vq'].append(val_vq)
        history['val_cosine'].append(val_cosine)

        # Update epoch progress bar
        epoch_pbar.set_postfix({
            'train_loss': f"{train_loss:.4f}",
            'val_loss': f"{val_loss:.4f}",
            'cosine': f"{val_cosine:.4f}"
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), '/content/drive/MyDrive/semantic_wm/models/best_vqvae_model.pth')

    return history


print("✅ Training 함수 정의 완료")
print("   - train_vae: VAE 학습")
print("   - train_vqvae: VQ-VAE 학습")

## 14. VAE 모델 학습

In [ ]:
# VAE 모델 초기화
vae_model = ImprovedVAE(input_dim=512, latent_dim=100).to(device)

print("="*80)
print("VAE 모델 학습 시작")
print("="*80)
print(f"Model: ImprovedVAE (512D → 100D)")
print(f"Beta: 0.01 (작은 값으로 클러스터 유지)")
print(f"Epochs: 50")
print(f"Learning rate: 1e-3")
print("="*80)

# 학습 시작
vae_history = train_vae(
    model=vae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    lr=1e-3,
    beta=0.01
)

print("\n" + "="*80)
print("VAE 학습 완료!")
print("="*80)
print(f"Best Val Loss: {min(vae_history['val_losses']):.4f}")
print(f"Best Val Cosine: {max(vae_history['val_cosine']):.4f}")
print("="*80)

## 14-1. VAE Loss 시각화

In [ ]:
# VAE Loss 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Total Loss
axes[0, 0].plot(vae_history['train_losses'], label='Train', linewidth=2, color='#3498db')
axes[0, 0].plot(vae_history['val_losses'], label='Val', linewidth=2, color='#e74c3c')
axes[0, 0].set_title('VAE Total Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Reconstruction Loss
axes[0, 1].plot(vae_history['train_recon'], label='Train', linewidth=2, color='#2ecc71')
axes[0, 1].plot(vae_history['val_recon'], label='Val', linewidth=2, color='#e67e22')
axes[0, 1].set_title('Reconstruction Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. KLD Loss
axes[1, 0].plot(vae_history['train_kld'], label='Train', linewidth=2, color='#9b59b6')
axes[1, 0].plot(vae_history['val_kld'], label='Val', linewidth=2, color='#34495e')
axes[1, 0].set_title('KLD Loss (β=0.01)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Cosine Similarity
axes[1, 1].plot(vae_history['val_cosine'], linewidth=2, color='#1abc9c', marker='o', markersize=3)
axes[1, 1].fill_between(range(len(vae_history['val_cosine'])),
                         vae_history['val_cosine'], alpha=0.3, color='#1abc9c')
axes[1, 1].set_title('Validation Cosine Similarity', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Cosine Similarity')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/vae_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ VAE Loss 시각화 완료")
print(f"   - Final Train Loss: {vae_history['train_losses'][-1]:.4f}")
print(f"   - Final Val Loss: {vae_history['val_losses'][-1]:.4f}")
print(f"   - Final Cosine Sim: {vae_history['val_cosine'][-1]:.4f}")

## 15. VQ-VAE 모델 학습

In [ ]:
# VQ-VAE 모델 초기화
vqvae_model = ImprovedVQVAE(input_dim=512, latent_dim=100, num_embeddings=512).to(device)

print("="*80)
print("VQ-VAE 모델 학습 시작")
print("="*80)
print(f"Model: ImprovedVQVAE (512D → 100D)")
print(f"Codebook: 512 entries")
print(f"Epochs: 50")
print(f"Learning rate: 1e-3")
print("="*80)

# 학습 시작
vqvae_history = train_vqvae(
    model=vqvae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    lr=1e-3
)

print("\n" + "="*80)
print("VQ-VAE 학습 완료!")
print("="*80)
print(f"Best Val Loss: {min(vqvae_history['val_losses']):.4f}")
print(f"Best Val Cosine: {max(vqvae_history['val_cosine']):.4f}")
print("="*80)

## 15-1. VQ-VAE Loss 시각화

In [ ]:
# VQ-VAE Loss 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Total Loss
axes[0, 0].plot(vqvae_history['train_losses'], label='Train', linewidth=2, color='#3498db')
axes[0, 0].plot(vqvae_history['val_losses'], label='Val', linewidth=2, color='#e74c3c')
axes[0, 0].set_title('VQ-VAE Total Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Reconstruction Loss
axes[0, 1].plot(vqvae_history['train_recon'], label='Train', linewidth=2, color='#2ecc71')
axes[0, 1].plot(vqvae_history['val_recon'], label='Val', linewidth=2, color='#e67e22')
axes[0, 1].set_title('Reconstruction Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. VQ Loss
axes[1, 0].plot(vqvae_history['train_vq'], label='Train', linewidth=2, color='#9b59b6')
axes[1, 0].plot(vqvae_history['val_vq'], label='Val', linewidth=2, color='#34495e')
axes[1, 0].set_title('VQ Loss (Codebook + Commitment)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Cosine Similarity
axes[1, 1].plot(vqvae_history['val_cosine'], linewidth=2, color='#1abc9c', marker='o', markersize=3)
axes[1, 1].fill_between(range(len(vqvae_history['val_cosine'])),
                         vqvae_history['val_cosine'], alpha=0.3, color='#1abc9c')
axes[1, 1].set_title('Validation Cosine Similarity', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Cosine Similarity')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/vqvae_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ VQ-VAE Loss 시각화 완료")
print(f"   - Final Train Loss: {vqvae_history['train_losses'][-1]:.4f}")
print(f"   - Final Val Loss: {vqvae_history['val_losses'][-1]:.4f}")
print(f"   - Final Cosine Sim: {vqvae_history['val_cosine'][-1]:.4f}")

## 16. Latent Space 및 Reconstructed CLIP t-SNE 시각화

In [ ]:
from sklearn.manifold import TSNE

# Test 데이터로 latent 및 reconstruction 추출
test_embeddings_tensor = torch.FloatTensor(test_embeddings).to(device)

# VAE
vae_model.eval()
with torch.no_grad():
    vae_latents = vae_model.get_latent(test_embeddings_tensor).cpu().numpy()
    vae_recon, _, _, _ = vae_model(test_embeddings_tensor)
    vae_recon = vae_recon.cpu().numpy()

# VQ-VAE
vqvae_model.eval()
with torch.no_grad():
    vqvae_latents, _ = vqvae_model.get_latent(test_embeddings_tensor)
    vqvae_latents = vqvae_latents.cpu().numpy()
    vqvae_recon, _, _ = vqvae_model(test_embeddings_tensor)
    vqvae_recon = vqvae_recon.cpu().numpy()

print("✅ Latent 및 Reconstruction 추출 완료")
print(f"   - Original CLIP: {test_embeddings.shape}")
print(f"   - VAE Latent: {vae_latents.shape}")
print(f"   - VAE Recon: {vae_recon.shape}")
print(f"   - VQ-VAE Latent: {vqvae_latents.shape}")
print(f"   - VQ-VAE Recon: {vqvae_recon.shape}")

In [ ]:
# t-SNE 계산
print("\nt-SNE 계산 중...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)

original_2d = tsne.fit_transform(test_embeddings)
vae_latent_2d = tsne.fit_transform(vae_latents)
vae_recon_2d = tsne.fit_transform(vae_recon)
vqvae_latent_2d = tsne.fit_transform(vqvae_latents)
vqvae_recon_2d = tsne.fit_transform(vqvae_recon)

print("✅ t-SNE 계산 완료")

In [ ]:
# 시각화
colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

fig, axes = plt.subplots(2, 3, figsize=(20, 13))

# Row 1: Original, VAE Latent, VAE Recon
data_row1 = [
    (original_2d, "Original CLIP (512D)", axes[0, 0]),
    (vae_latent_2d, "VAE Latent Space (100D)", axes[0, 1]),
    (vae_recon_2d, "VAE Reconstructed CLIP (512D)", axes[0, 2])
]

# Row 2: Original, VQ-VAE Latent, VQ-VAE Recon
data_row2 = [
    (original_2d, "Original CLIP (512D)", axes[1, 0]),
    (vqvae_latent_2d, "VQ-VAE Latent Space (100D)", axes[1, 1]),
    (vqvae_recon_2d, "VQ-VAE Reconstructed CLIP (512D)", axes[1, 2])
]

# Plot Row 1 (VAE)
for data_2d, title, ax in data_row1:
    for category in categories:
        mask = test_labels == category
        ax.scatter(
            data_2d[mask, 0], data_2d[mask, 1],
            c=colors[category], marker=markers[category],
            label=f'{category.capitalize()} ({np.sum(mask)})',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')

# Plot Row 2 (VQ-VAE)
for data_2d, title, ax in data_row2:
    for category in categories:
        mask = test_labels == category
        ax.scatter(
            data_2d[mask, 0], data_2d[mask, 1],
            c=colors[category], marker=markers[category],
            label=f'{category.capitalize()} ({np.sum(mask)})',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')

plt.suptitle('VAE vs VQ-VAE: Latent Space & Reconstructed CLIP Comparison',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/latent_reconstruction_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ t-SNE 시각화 완료")
print("   - Row 1: VAE (Original → Latent → Reconstructed)")
print("   - Row 2: VQ-VAE (Original → Latent → Reconstructed)")

In [ ]:
# Watermark 변환 클래스 (Latent ↔ Watermark)
class LatentWatermarker:
    """
    Latent vector를 Watermark (bit array)로 변환하고 복원하는 클래스
    실제 이미지 삽입을 시뮬레이션하기 위해 quantization 수행
    """
    def __init__(self, latent_dim=100):
        """
        Args:
            latent_dim: Latent vector 차원 (100D)
            bits_per_value: 각 latent 값당 비트 수 (1 = sign bit만)
        """
        self.latent_dim = latent_dim

    def compute_latent_stats(self, latent_vectors):
      """
      latent vectors: shape(N, latent_dim)의 numpy array
      """
      return {
          'mean': latent_vectors.mean(axis=0),
          'std': latent_vectors.std(axis=0)
      }

    def latent_to_watermark(self, latent_vector):
        """
        latent_vector: shape (latent_dim,) numpy array
        Returns:
            watermark: 0/1 binary array, shape (latent_dim,)
        """
        watermark = (latent_vector > 0).astype(np.int32)

        return watermark

    def watermark_to_latent(self, watermark, latent_stats):
        """
        watermark: 0/1 array, shape (latent_dim,)
        latent_stats: {'mean': [...], 'std': [...]}

        Returns:
            restored_latent: shape (latent_dim,) numpy array
        """
        # 1) 0/1 → -1/+1 (sign 복원)
        restored_latent = watermark.astype(np.float32) * 2 - 1  # {0→-1, 1→+1}

        # 2) mean/std 기반 scale 복원 (semantic 보존에 가장 중요!)
        if latent_stats is not None:
            mean = latent_stats['mean']
            std = latent_stats['std']

            # 원래 latent의 geometry를 유지하는 핵심
            restored_latent = restored_latent * std + mean

        return restored_latent

# Watermarker 초기화
watermarker = LatentWatermarker(latent_dim=100)

print("="*80)
print("✅ Watermark 변환 클래스 정의 완료")
print("="*80)
print(f"   Latent dimension: {watermarker.latent_dim}")
print("="*80)

## 17. Test 데이터셋 평가: VAE vs VQ-VAE 비교

In [ ]:
# Test 데이터 준비 (Watermark Pipeline)
test_embeddings_tensor = torch.FloatTensor(test_embeddings).to(device)

print("="*80)
print("Test 데이터셋 평가: VAE vs VQ-VAE (Watermark Pipeline)")
print("="*80)
print(f"Test samples: {len(test_embeddings)}")
print(f"Categories: {categories}")
print(f"Pipeline: Original → Encoder → Latent → Watermark → Latent → Decoder → Reconstructed")
print("="*80)

# ========================================
# VAE: Watermark Pipeline
# ========================================
print("\n" + "="*80)
print("[VAE] Watermark Pipeline 시작")
print("="*80)

vae_model.eval()

# Step 1: Latent 추출 (Encoder)
print("  [1/5] Latent 추출 중...")
with torch.no_grad():
    mu, logvar = vae_model.encode(test_embeddings_tensor)
    vae_test_latents = mu
vae_test_latents_np = vae_test_latents.cpu().numpy()
print(f"        ✓ Latent shape: {vae_test_latents_np.shape}")

# Step 2: Latent 통계 계산
print("  [2/5] Latent 통계 계산 중...")
vae_latent_stats = watermarker.compute_latent_stats(vae_test_latents_np)
print(f"        ✓ Mean: {vae_latent_stats['mean'][:5]} ...")
print(f"        ✓ Std: {vae_latent_stats['std'][:5]} ...")

# Step 3: Watermark 생성
print("  [3/5] Watermark 생성 중...")
vae_watermarks = np.array([watermarker.latent_to_watermark(latent)
                            for latent in vae_test_latents_np])
print(f"        ✓ Watermark shape: {vae_watermarks.shape}")

# Step 4: Latent 복원
print("  [4/5] Latent 복원 중...")
vae_test_latents_restored = np.array([watermarker.watermark_to_latent(wm, vae_latent_stats)
                                       for wm in vae_watermarks])
vae_latents_restored_tensor = torch.FloatTensor(vae_test_latents_restored).to(device)
print(f"        ✓ Restored latent shape: {vae_test_latents_restored.shape}")

# Step 5: CLIP 복원
print("  [5/5] CLIP 복원 중...")
with torch.no_grad():
    vae_test_recon = vae_model.decode(vae_latents_restored_tensor)
vae_test_recon = vae_test_recon.cpu().numpy()
print(f"        ✓ Reconstructed CLIP shape: {vae_test_recon.shape}")

print("\n✅ VAE Watermark Pipeline 완료")
print(f"   - Original Latent: {vae_test_latents_np.shape}")
print(f"   - Watermark: {vae_watermarks.shape} (100-bit)")
print(f"   - Restored Latent: {vae_test_latents_restored.shape}")
print(f"   - Reconstructed CLIP: {vae_test_recon.shape}")

# ========================================
# VQ-VAE: Watermark Pipeline
# ========================================
print("\n" + "="*80)
print("[VQ-VAE] Watermark Pipeline 시작")
print("="*80)

vqvae_model.eval()

# Step 1: Latent 추출 (Encoder + Quantize)
print("  [1/5] Latent 추출 중...")
with torch.no_grad():
    vqvae_test_latents, vqvae_test_indices = vqvae_model.get_latent(test_embeddings_tensor)
vqvae_test_latents_np = vqvae_test_latents.cpu().numpy()
vqvae_test_indices = vqvae_test_indices.cpu().numpy()
print(f"        ✓ Latent shape: {vqvae_test_latents_np.shape}")
print(f"        ✓ Codebook indices: {vqvae_test_indices.shape}")

# Step 2: Latent 통계 계산
print("  [2/5] Latent 통계 계산 중...")
vqvae_latent_stats = watermarker.compute_latent_stats(vqvae_test_latents_np)
print(f"        ✓ Mean: {vqvae_latent_stats['mean'][:5]} ...")
print(f"        ✓ Std: {vqvae_latent_stats['std'][:5]} ...")

# Step 3: Watermark 생성
print("  [3/5] Watermark 생성 중...")
vqvae_watermarks = np.array([watermarker.latent_to_watermark(latent)
                              for latent in vqvae_test_latents_np])
print(f"        ✓ Watermark shape: {vqvae_watermarks.shape}")

# Step 4: Latent 복원
print("  [4/5] Latent 복원 중...")
vqvae_test_latents_restored = np.array([watermarker.watermark_to_latent(wm, vqvae_latent_stats)
                                         for wm in vqvae_watermarks])
vqvae_latents_restored_tensor = torch.FloatTensor(vqvae_test_latents_restored).to(device)
print(f"        ✓ Restored latent shape: {vqvae_test_latents_restored.shape}")

# Step 5: CLIP 복원
print("  [5/5] CLIP 복원 중...")
with torch.no_grad():
    vqvae_test_recon = vqvae_model.decode(vqvae_latents_restored_tensor)
vqvae_test_recon = vqvae_test_recon.cpu().numpy()
print(f"        ✓ Reconstructed CLIP shape: {vqvae_test_recon.shape}")

print("\n✅ VQ-VAE Watermark Pipeline 완료")
print(f"   - Original Latent: {vqvae_test_latents_np.shape}")
print(f"   - Watermark: {vqvae_watermarks.shape} (100-bit)")
print(f"   - Restored Latent: {vqvae_test_latents_restored.shape}")
print(f"   - Reconstructed CLIP: {vqvae_test_recon.shape}")

print("\n" + "="*80)
print("✅ VAE vs VQ-VAE Watermark Pipeline 완료")
print("="*80)

In [ ]:
# Cosine Similarity 계산 (Watermark Pipeline)
print("\n" + "="*80)
print("성능 평가: Cosine Similarity & Latent Recovery")
print("="*80)

# ========================================
# VAE 성능 평가
# ========================================
# CLIP 복원 성능
vae_cosine_sims = F.cosine_similarity(
    torch.FloatTensor(test_embeddings),
    torch.FloatTensor(vae_test_recon),
    dim=1
).numpy()

# Latent 복원 정확도
vae_latent_cosine = F.cosine_similarity(
    torch.FloatTensor(vae_test_latents_np),
    torch.FloatTensor(vae_test_latents_restored),
    dim=1
).numpy()

vae_cosine_dist = 1 - vae_cosine_sims

# ========================================
# VQ-VAE 성능 평가
# ========================================
# CLIP 복원 성능
vqvae_cosine_sims = F.cosine_similarity(
    torch.FloatTensor(test_embeddings),
    torch.FloatTensor(vqvae_test_recon),
    dim=1
).numpy()

# Latent 복원 정확도
vqvae_latent_cosine = F.cosine_similarity(
    torch.FloatTensor(vqvae_test_latents_np),
    torch.FloatTensor(vqvae_test_latents_restored),
    dim=1
).numpy()

vqvae_cosine_dist = 1 - vqvae_cosine_sims

# ========================================
# 전체 통계 출력
# ========================================
print(f"\n📊 전체 성능 통계 (Watermark Pipeline):")
print(f"\n{'='*80}")
print(f"{'Model':<12} {'CLIP Cosine':<15} {'Latent Recovery':<18} {'CLIP Dist':<12}")
print(f"{'='*80}")
print(f"{'VAE':<12} {vae_cosine_sims.mean():<15.6f} {vae_latent_cosine.mean():<18.6f} {vae_cosine_dist.mean():<12.6f}")
print(f"{'VQ-VAE':<12} {vqvae_cosine_sims.mean():<15.6f} {vqvae_latent_cosine.mean():<18.6f} {vqvae_cosine_dist.mean():<12.6f}")
print(f"{'='*80}")

print(f"\n[VAE 상세]")
print(f"   ├─ CLIP Cosine Sim: {vae_cosine_sims.mean():.6f} ± {vae_cosine_sims.std():.6f}")
print(f"   ├─ Latent Recovery: {vae_latent_cosine.mean():.6f} ± {vae_latent_cosine.std():.6f}")
print(f"   ├─ CLIP Cosine Dist: {vae_cosine_dist.mean():.6f}")
print(f"   └─ Min/Max Sim: [{vae_cosine_sims.min():.6f}, {vae_cosine_sims.max():.6f}]")

print(f"\n[VQ-VAE 상세]")
print(f"   ├─ CLIP Cosine Sim: {vqvae_cosine_sims.mean():.6f} ± {vqvae_cosine_sims.std():.6f}")
print(f"   ├─ Latent Recovery: {vqvae_latent_cosine.mean():.6f} ± {vqvae_latent_cosine.std():.6f}")
print(f"   ├─ CLIP Cosine Dist: {vqvae_cosine_dist.mean():.6f}")
print(f"   └─ Min/Max Sim: [{vqvae_cosine_sims.min():.6f}, {vqvae_cosine_sims.max():.6f}]")

# ========================================
# 카테고리별 통계
# ========================================
print(f"\n📈 카테고리별 통계 (Watermark Pipeline):")
print(f"{'='*80}")
for category in categories:
    mask = test_labels == category
    vae_cat_sims = vae_cosine_sims[mask]
    vqvae_cat_sims = vqvae_cosine_sims[mask]
    vae_cat_latent = vae_latent_cosine[mask]
    vqvae_cat_latent = vqvae_latent_cosine[mask]

    print(f"\n[{category.capitalize()} - n={np.sum(mask)}]")
    print(f"   {'Model':<12} {'CLIP Cosine':<18} {'Latent Recovery':<18}")
    print(f"   {'-'*50}")
    print(f"   {'VAE':<12} {vae_cat_sims.mean():<18.6f} {vae_cat_latent.mean():<18.6f}")
    print(f"   {'VQ-VAE':<12} {vqvae_cat_sims.mean():<18.6f} {vqvae_cat_latent.mean():<18.6f}")

# ========================================
# 승자 판정
# ========================================
print(f"\n{'='*80}")
print("🏆 최종 승자 판정 (Watermark Pipeline)")
print(f"{'='*80}")

if vae_cosine_sims.mean() > vqvae_cosine_sims.mean():
    clip_winner = "VAE"
    clip_diff = vae_cosine_sims.mean() - vqvae_cosine_sims.mean()
else:
    clip_winner = "VQ-VAE"
    clip_diff = vqvae_cosine_sims.mean() - vae_cosine_sims.mean()

if vae_latent_cosine.mean() > vqvae_latent_cosine.mean():
    latent_winner = "VAE"
    latent_diff = vae_latent_cosine.mean() - vqvae_latent_cosine.mean()
else:
    latent_winner = "VQ-VAE"
    latent_diff = vqvae_latent_cosine.mean() - vae_latent_cosine.mean()

print(f"\n✅ CLIP 복원 성능: {clip_winner} 승리")
print(f"   → 차이: {clip_diff:.6f}")
print(f"   → VAE: {vae_cosine_sims.mean():.6f}")
print(f"   → VQ-VAE: {vqvae_cosine_sims.mean():.6f}")

print(f"\n✅ Latent 복원 정확도: {latent_winner} 승리")
print(f"   → 차이: {latent_diff:.6f}")
print(f"   → VAE: {vae_latent_cosine.mean():.6f}")
print(f"   → VQ-VAE: {vqvae_latent_cosine.mean():.6f}")

print(f"\n💡 해석:")
print(f"   - Latent Recovery: Watermark(100-bit) → Latent 복원 정확도")
print(f"   - CLIP Cosine: 최종 CLIP embedding 복원 품질")
print(f"   - Watermark 변환 과정에서 정보 손실이 발생함")
print("="*80)

In [ ]:
# t-SNE 시각화: Original, VAE Recon, VQ-VAE Recon (Watermark Pipeline)
print("\n" + "="*80)
print("t-SNE 시각화 준비 중... (Watermark Pipeline 결과)")
print("="*80)

from sklearn.manifold import TSNE

# t-SNE 계산
print("\n[1/5] 원본 CLIP t-SNE 계산...")
tsne_original = TSNE(n_components=2, random_state=42, perplexity=30)
original_2d = tsne_original.fit_transform(test_embeddings)
print(f"      ✓ Original shape: {original_2d.shape}")

print("[2/5] VAE Latent t-SNE 계산...")
tsne_vae_latent = TSNE(n_components=2, random_state=42, perplexity=30)
vae_latent_2d = tsne_vae_latent.fit_transform(vae_test_latents_np)
print(f"      ✓ VAE Latent shape: {vae_latent_2d.shape}")

print("[3/5] VAE Reconstructed CLIP t-SNE 계산...")
tsne_vae = TSNE(n_components=2, random_state=42, perplexity=30)
vae_recon_2d = tsne_vae.fit_transform(vae_test_recon)
print(f"      ✓ VAE Recon shape: {vae_recon_2d.shape}")

print("[4/5] VQ-VAE Latent t-SNE 계산...")
tsne_vqvae_latent = TSNE(n_components=2, random_state=42, perplexity=30)
vqvae_latent_2d = tsne_vqvae_latent.fit_transform(vqvae_test_latents_np)
print(f"      ✓ VQ-VAE Latent shape: {vqvae_latent_2d.shape}")

print("[5/5] VQ-VAE Reconstructed CLIP t-SNE 계산...")
tsne_vqvae = TSNE(n_components=2, random_state=42, perplexity=30)
vqvae_recon_2d = tsne_vqvae.fit_transform(vqvae_test_recon)
print(f"      ✓ VQ-VAE Recon shape: {vqvae_recon_2d.shape}")

print("\n✅ 모든 t-SNE 계산 완료")

print("✅ t-SNE 계산 완료")

# 시각화
colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

titles = [
    'Original CLIP (Test Set)',
    'VAE Reconstructed CLIP',
    'VQ-VAE Reconstructed CLIP'
]
data_list = [original_2d, vae_recon_2d, vqvae_recon_2d]

for ax, title, data_2d in zip(axes, titles, data_list):
    for category in categories:
        mask = test_labels == category
        ax.scatter(
            data_2d[mask, 0], data_2d[mask, 1],
            c=colors[category], marker=markers[category],
            label=f'{category.capitalize()} ({np.sum(mask)})',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')

plt.suptitle('Test Dataset: Original vs VAE vs VQ-VAE Reconstructed CLIP',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/test_comparison_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ t-SNE 시각화 완료")
print(f"   저장 위치: /content/test_comparison_tsne.png")
print("="*80)

In [ ]:
# Cosine Distance Distribution 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 전체 Cosine Distance 분포 비교
ax = axes[0, 0]
ax.hist(vae_cosine_dist, bins=50, alpha=0.6, color='#3498db',
        label=f'VAE (mean={vae_cosine_dist.mean():.4f})', edgecolor='black')
ax.hist(vqvae_cosine_dist, bins=50, alpha=0.6, color='#e74c3c',
        label=f'VQ-VAE (mean={vqvae_cosine_dist.mean():.4f})', edgecolor='black')
ax.axvline(vae_cosine_dist.mean(), color='#3498db', linestyle='--', linewidth=2)
ax.axvline(vqvae_cosine_dist.mean(), color='#e74c3c', linestyle='--', linewidth=2)
ax.set_xlabel('Cosine Distance (1 - Cosine Similarity)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Overall Cosine Distance Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. 전체 Cosine Similarity 분포 비교
ax = axes[0, 1]
ax.hist(vae_cosine_sims, bins=50, alpha=0.6, color='#2ecc71',
        label=f'VAE (mean={vae_cosine_sims.mean():.4f})', edgecolor='black')
ax.hist(vqvae_cosine_sims, bins=50, alpha=0.6, color='#e67e22',
        label=f'VQ-VAE (mean={vqvae_cosine_sims.mean():.4f})', edgecolor='black')
ax.axvline(vae_cosine_sims.mean(), color='#2ecc71', linestyle='--', linewidth=2)
ax.axvline(vqvae_cosine_sims.mean(), color='#e67e22', linestyle='--', linewidth=2)
ax.set_xlabel('Cosine Similarity', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Overall Cosine Similarity Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 3. 카테고리별 Cosine Distance (VAE)
ax = axes[1, 0]
for category in categories:
    mask = test_labels == category
    ax.hist(vae_cosine_dist[mask], bins=30, alpha=0.5,
            label=f'{category.capitalize()}', color=colors[category], edgecolor='black')
ax.set_xlabel('Cosine Distance', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('VAE: Category-wise Cosine Distance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 4. 카테고리별 Cosine Distance (VQ-VAE)
ax = axes[1, 1]
for category in categories:
    mask = test_labels == category
    ax.hist(vqvae_cosine_dist[mask], bins=30, alpha=0.5,
            label=f'{category.capitalize()}', color=colors[category], edgecolor='black')
ax.set_xlabel('Cosine Distance', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('VQ-VAE: Category-wise Cosine Distance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Test Dataset: VAE vs VQ-VAE Cosine Distance/Similarity Distributions',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/test_cosine_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Cosine Distance/Similarity 분포 시각화 완료")
print(f"   저장 위치: /content/test_cosine_distributions.png")

In [ ]:
# 카테고리별 Box Plot 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cosine Similarity Box Plot
ax = axes[0]
data_to_plot = []
labels_plot = []
for category in categories:
    mask = test_labels == category
    data_to_plot.append(vae_cosine_sims[mask])
    labels_plot.append(f'VAE-{category[:3]}')
for category in categories:
    mask = test_labels == category
    data_to_plot.append(vqvae_cosine_sims[mask])
    labels_plot.append(f'VQ-{category[:3]}')

bp = ax.boxplot(data_to_plot, labels=labels_plot, patch_artist=True)
for i, patch in enumerate(bp['boxes']):
    if i < 3:  # VAE
        patch.set_facecolor('#3498db')
        patch.set_alpha(0.6)
    else:  # VQ-VAE
        patch.set_facecolor('#e74c3c')
        patch.set_alpha(0.6)

ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Category-wise Cosine Similarity Comparison', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)

# Cosine Distance Box Plot
ax = axes[1]
data_to_plot = []
for category in categories:
    mask = test_labels == category
    data_to_plot.append(vae_cosine_dist[mask])
for category in categories:
    mask = test_labels == category
    data_to_plot.append(vqvae_cosine_dist[mask])

bp = ax.boxplot(data_to_plot, labels=labels_plot, patch_artist=True)
for i, patch in enumerate(bp['boxes']):
    if i < 3:  # VAE
        patch.set_facecolor('#2ecc71')
        patch.set_alpha(0.6)
    else:  # VQ-VAE
        patch.set_facecolor('#e67e22')
        patch.set_alpha(0.6)

ax.set_ylabel('Cosine Distance', fontsize=12)
ax.set_title('Category-wise Cosine Distance Comparison', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/content/test_boxplot_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Box Plot 비교 시각화 완료")
print(f"   저장 위치: /content/test_boxplot_comparison.png")

In [ ]:
# 종합 분석 및 결론
print("\n" + "="*80)
print("TEST 데이터셋 종합 분석 결과: VAE vs VQ-VAE")
print("="*80)

print("\n📊 전체 성능 비교:")
print(f"\n[VAE]")
print(f"   • 평균 Cosine Similarity: {vae_cosine_sims.mean():.6f} ± {vae_cosine_sims.std():.6f}")
print(f"   • 평균 Cosine Distance: {vae_cosine_dist.mean():.6f} ± {vae_cosine_dist.std():.6f}")
print(f"   • Min/Max Similarity: [{vae_cosine_sims.min():.6f}, {vae_cosine_sims.max():.6f}]")

print(f"\n[VQ-VAE]")
print(f"   • 평균 Cosine Similarity: {vqvae_cosine_sims.mean():.6f} ± {vqvae_cosine_sims.std():.6f}")
print(f"   • 평균 Cosine Distance: {vqvae_cosine_dist.mean():.6f} ± {vqvae_cosine_dist.std():.6f}")
print(f"   • Min/Max Similarity: [{vqvae_cosine_sims.min():.6f}, {vqvae_cosine_sims.max():.6f}]")

# 성능 차이 계산
sim_diff = vae_cosine_sims.mean() - vqvae_cosine_sims.mean()
dist_diff = vae_cosine_dist.mean() - vqvae_cosine_dist.mean()

print(f"\n[차이]")
if sim_diff > 0:
    print(f"   • VAE가 {abs(sim_diff):.6f} 더 높은 Cosine Similarity ✅")
else:
    print(f"   • VQ-VAE가 {abs(sim_diff):.6f} 더 높은 Cosine Similarity ✅")

if dist_diff < 0:
    print(f"   • VAE가 {abs(dist_diff):.6f} 더 낮은 Cosine Distance ✅")
else:
    print(f"   • VQ-VAE가 {abs(dist_diff):.6f} 더 낮은 Cosine Distance ✅")

print("\n📈 카테고리별 성능 비교:")
for category in categories:
    mask = test_labels == category
    vae_cat_mean = vae_cosine_sims[mask].mean()
    vqvae_cat_mean = vqvae_cosine_sims[mask].mean()
    vae_cat_std = vae_cosine_sims[mask].std()
    vqvae_cat_std = vqvae_cosine_sims[mask].std()

    print(f"\n[{category.capitalize()}] ({np.sum(mask)} samples)")
    print(f"   VAE:    {vae_cat_mean:.6f} ± {vae_cat_std:.6f}")
    print(f"   VQ-VAE: {vqvae_cat_mean:.6f} ± {vqvae_cat_std:.6f}")

    cat_diff = vae_cat_mean - vqvae_cat_mean
    if cat_diff > 0:
        print(f"   → VAE가 {abs(cat_diff):.6f} 더 우수 ✅")
    else:
        print(f"   → VQ-VAE가 {abs(cat_diff):.6f} 더 우수 ✅")

print("\n💡 해석:")
if vae_cosine_sims.mean() > 0.95 and vqvae_cosine_sims.mean() > 0.95:
    print("   ✅ 두 모델 모두 매우 우수한 복원 품질 (Cosine Sim > 0.95)")
    print("      → Watermark에서 원본 CLIP의 semantic 정보를 거의 완벽하게 보존")
elif vae_cosine_sims.mean() > 0.90 or vqvae_cosine_sims.mean() > 0.90:
    print("   ✅ 우수한 복원 품질 (Cosine Sim > 0.90)")
    print("      → Watermark가 원본 CLIP의 semantic 정보를 잘 보존")
else:
    print("   ⚠️ 복원 품질 개선 필요")
    print("      → Beta 값 조정 또는 모델 구조 개선 검토")

# 승자 판정
print("\n🏆 최종 결과:")
if abs(sim_diff) < 0.001:
    print("   두 모델의 성능이 거의 동일합니다 (차이 < 0.001)")
elif sim_diff > 0:
    print(f"   VAE가 VQ-VAE보다 우수합니다 (Δ={sim_diff:.6f})")
    print("   → Continuous latent space가 더 효과적")
else:
    print(f"   VQ-VAE가 VAE보다 우수합니다 (Δ={abs(sim_diff):.6f})")
    print("   → Discrete latent space (codebook)가 더 효과적")

print("\n✅ Test 데이터셋 평가 완료!")
print("="*80)